# Practical Mini-projects

## Auto-cleanup and Archiving Script

In [1]:
import os, shutil, zipfile
from datetime import datetime, timedelta

class LogArchiver:
    def __init__(self, log_root, archive_folder, delete_after_days=30, archive_within_days=7):
        self.log_root = log_root
        self.archive_folder = archive_folder
        self.delete_after_days = delete_after_days
        self.archive_within_days  = archive_within_days
        self.files_to_archive = []
        os.makedirs(self.archive_folder, exist_ok=True)
    
    def scan_logs(self):
        print(f"Scanning {self.log_root} for log files...")
        for root, dirs, files in os.walk(self.log_root):
            for file in files:
                if file.endswith(".log"):
                    file_path = os.path.join(root, file)
                    self.process_file(file_path)
        if self.files_to_archive:
            self.archive_all()
    
    def process_file(self, file_path):
        stats_info = os.stat(file_path)
        file_mtime = datetime.fromtimestamp(stats_info.st_mtime)
        now = datetime.now()
        age_days = (now - file_mtime).days
        if age_days > self.delete_after_days:
            self.delete_file(file_path)
        elif age_days <= self.archive_within_days:
            self.files_to_archive.append(file_path)
    
    def delete_file(self, file_path):
        try:
            os.remove(file_path)
            print(f"Deleted old log: {file_path}")
        except Exception as e:
            print(f"Could not delete {file_path}: {e}")
    
    def archive_all(self):
        timestamp = datetime.now().strftime("%Y-%m-%d")
        archive_name = os.path.join(self.archive_folder, f"shipment_logs_{timestamp}.zip")
        try:
            with zipfile.ZipFile(archive_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
                for file_path in self.files_to_archive:
                    arcname = os.path.relpath(file_path, self.log_root)
                    zipf.write(file_path, arcname=arcname)
                    print(f"Added to archive: {file_path}")
                    os.remove(file_path)
            print(f"All recent logs archived to: {archive_name}")
        except Exception as e:
            print(f"Could not create archive: {e}")

def simulate_log_environment(log_dir):
    os.makedirs(log_dir, exist_ok=True)
    now = datetime.now()
    for i in range(10):
        age = timedelta(days=i*5)
        file_path = os.path.join(log_dir, f"shipment_log_{i}.log")
        with open(file_path, 'w') as f:
            f.write("Log entry...\n" * 5)
        old_time = now - age
        mod_time = old_time.timestamp()
        os.utime(file_path, (mod_time, mod_time))
    print(f"Simulated 10 dummy log files in {log_dir}")

if __name__ == "__main__":
    log_directory  = "logs"
    archive_output = "archived_logs"

    archiver = LogArchiver(
        log_root=log_directory,
        archive_folder=archive_output,
        delete_after_days=30,
        archive_within_days=7
    )

    archiver.scan_logs()

Scanning logs for log files...


## File Searching and Keyword Matching with Globbing

In [2]:
import glob
from pathlib import Path

class KeywordScanner:
    def __init__(self, root_dir, pattern="*.md", keywords=None):
        self.root_dir = root_dir
        self.pattern = pattern
        self.keywords = keywords or ["TODO"]
        self.matched_files = []
    
    def scan_with_os_walk(self):
        print(f"Searching with os.walk for {self.pattern} in {self.root_dir}")
        for dirpath, _, filenames in os.walk(self.root_dir):
            for filename in filenames:
                if glob.fnmatch.fnmatch(filename, self.pattern):
                    full_path = os.path.join(dirpath, filename)
                    if self.search_file(full_path):
                        self.matched_files.append(full_path)
    
    def scan_with_pathlib(self):
        print(f"Searching with Path.rglob for {self.pattern}")
        for file_path in Path(self.root_dir).rglob(self.pattern):
            if file_path.is_file() and self.search_file(file_path):
                self.matched_files.append(str(file_path))
    
    def search_file(self, filepath):
        try:
            with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
                content = f.read()
                for keyword in self.keywords:
                    if keyword in self.keywords:
                        if keyword in content:
                            print(f"Match in {filepath} -> keyword: '{keyword}'")
                            return True
        except Exception as e:
            print(f"Could not read {filepath}: {e}")
        return False
    
    def report_results(self):
        print(f"Total matches found: {len(self.matched_files)}")
        for f in self.matched_files:
            print(f" - {f}")

def simulate_docs(root_dir):
    os.makedirs(root_dir, exist_ok=True)
    sample_data = [
        ("docs/team_notes.md", "Reminder: TODO add onboarding steps.\n"),
        ("docs/legal/privacy_policy.md", "Compliant with GDPR and other standards."),
        ("docs/archive/old_notes.txt", "Legacy TODOs to clean."),
        ("docs/dev/integration_guide.md", "Endpoints... TODO: Validate tokens."),
        ("docs/README.md", "Project README.\n")
    ]
    for path, content in sample_data:
        full_path = os.path.join(root_dir, path)
        os.makedirs(os.path.dirname(full_path), exist_ok=True)
        with open(full_path, 'w') as f:
            f.write(content)
    print(f"Simulated docs created in: {root_dir}")

if __name__ == "__main__":
    doc_dir = "project_docs"
    simulate_docs(doc_dir)
    scanner = KeywordScanner(
        root_dir=doc_dir,
        pattern="*.md",
        keywords=["TODO", "GDPR"]
    )

    scanner.scan_with_os_walk()
    scanner.scan_with_pathlib()
    scanner.report_results()

Simulated docs created in: project_docs
Searching with os.walk for *.md in project_docs
Match in project_docs/docs/team_notes.md -> keyword: 'TODO'
Match in project_docs/docs/dev/integration_guide.md -> keyword: 'TODO'
Match in project_docs/docs/legal/privacy_policy.md -> keyword: 'GDPR'
Searching with Path.rglob for *.md
Match in project_docs/docs/team_notes.md -> keyword: 'TODO'
Match in project_docs/docs/dev/integration_guide.md -> keyword: 'TODO'
Match in project_docs/docs/legal/privacy_policy.md -> keyword: 'GDPR'
Total matches found: 6
 - project_docs/docs/team_notes.md
 - project_docs/docs/dev/integration_guide.md
 - project_docs/docs/legal/privacy_policy.md
 - project_docs/docs/team_notes.md
 - project_docs/docs/dev/integration_guide.md
 - project_docs/docs/legal/privacy_policy.md


## JSON-based File Storage System

In [3]:
# pip install portalocker

In [4]:
import json, portalocker

class JSONKeyValueStore:
    def __init__ (self, filename):
        self.filename = filename
        self._ensure_file_exists()
    
    def _ensure_file_exists(self):
        os.makedirs(os.path.dirname(self.filename), exist_ok=True)
        if not os.path.exists(self.filename):
            with open(self.filename, 'w') as f:
                json.dump({}, f)
    
    def _read_store(self):
        with open(self.filename, 'r') as f:
            portalocker.lock(f, portalocker.LOCK_SH)
            data = json.load(f)
            portalocker.unlock(f)
        return data
    
    def _write_store(self, data):
        with open(self.filename, 'w') as f:
            portalocker.lock(f, portalocker.LOCK_EX)
            json.dump(data, f, indent=2)
            portalocker.unlock(f)
    
    def set(self, key, value):
        data  = self._read_store()
        data[key] = value
        self._write_store(data)
        print(f"Set key: '{key}' -> {value}")

    def get(self, key):
        data = self._read_store()
        value = data.get(key)
        print(f"Get key: '{key}' -> {value}")
        return value
    
    def delete(self, key):
        data = self._read_store()
        if key in data:
            del data[key]
            self._write_store(data)
            print(f"Deleted key: '{key}'")
        else:
            print(f"Key not found: '{key}'")

    def list_all(self):
        data = self._read_store()
        print(f"Current store contents:")
        for k, v in data.items():
            print(f" - {k}: {v}")
        return data
    
if __name__ == "__main__":
    store = JSONKeyValueStore("store/task_data.json")
    store.set("agent_42_todo", "Verify customs paperwork")
    store.set("agent_99_status", "Credentials expired")
    store.get("agent_42_todo")
    store.list_all()
    store.delete("agent_99_status")
    store.list_all()

Set key: 'agent_42_todo' -> Verify customs paperwork
Set key: 'agent_99_status' -> Credentials expired
Get key: 'agent_42_todo' -> Verify customs paperwork
Current store contents:
 - agent_42_todo: Verify customs paperwork
 - agent_99_status: Credentials expired
Deleted key: 'agent_99_status'
Current store contents:
 - agent_42_todo: Verify customs paperwork
